# Capa Gold: Modelo Dimensional y Conversión Multimoneda

**Objetivo:** Consumir las tablas procesadas de la capa Silver (`ventas_silver` y `tasas_cambio_silver`), unificar los montos en USD según la cotización del día y construir el Modelo Dimensional en Estrella (`fact_ventas`, `dim_clientes`, `dim_productos`).

**Salida:** `abfss://tpi-data@sttpiicaro.dfs.core.windows.net/gold/`

In [0]:
from pyspark.sql import functions as F

# -------------------------------------------------------------------------
# Conexión y Lectura
# -------------------------------------------------------------------------
STORAGE_ACCOUNT = "sttpiicaro"
CONTAINER = "tpi-data"
storage_key = "TU_CLAVE_AQUÍ"

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    storage_key
)

BASE = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# 1. Cargar DataFrames Silver
df_ventas_silver = spark.read.format("delta").load(f"{BASE}/silver/ventas_silver")
df_tasas_silver = spark.read.format("delta").load(f"{BASE}/silver/tasas_cambio_silver")

# 🔍 VISUALIZACIÓN 1: Ver esquemas de las tablas Silver
print("--- ESQUEMA: VENTAS SILVER ---")
df_ventas_silver.printSchema()

print("--- ESQUEMA: TASAS SILVER ---")
df_tasas_silver.printSchema()

# -------------------------------------------------------------------------
# 2. Cruce Multimoneda
# -------------------------------------------------------------------------
df_tasas_preparado = df_tasas_silver.withColumn("fecha_tasa", F.to_date(F.col("fecha_ingesta")))

df_enriquecido = (
    df_ventas_silver.alias("v")
    .join(
        df_tasas_preparado.alias("t"),
        (F.col("v.transaction_date") == F.col("t.fecha_tasa")) & 
        (F.col("v.currency") == F.col("t.moneda")),
        how="left"
    )
    .withColumn(
        "tasa_aplicada",
        F.when(F.col("v.currency") == "USD", 1.0)
         .otherwise(F.coalesce(F.col("t.tipo_cambio_usd"), F.lit(1.0)))
    )
    .withColumn(
        "total_amount_usd",
        F.when(F.col("v.currency") == "USD", F.col("v.total_amount_local"))
         .otherwise(F.round(F.col("v.total_amount_local") / F.col("tasa_aplicada"), 2))
    )
)

# 🔍 VISUALIZACIÓN 2: Ver vista previa del cálculo en USD
print("--- VISTA PREVIA DEL CRUCE Y MONTO EN USD ---")
display(df_enriquecido.select("v.transaction_id", "v.currency", "v.total_amount_local", "tasa_aplicada", "total_amount_usd").limit(5))

# -------------------------------------------------------------------------
# 3. Construcción y Guardado del Modelo Dimensional (Gold)
# -------------------------------------------------------------------------
dim_clientes = df_enriquecido.select(F.col("v.customer_id")).distinct()

columnas_prod = [c for c in ["product_id", "product_name", "category"] if c in df_enriquecido.columns]
dim_productos = df_enriquecido.select(*[F.col(f"v.{c}") for c in columnas_prod]).distinct()

fact_ventas = df_enriquecido.select(
    F.col("v.transaction_id"),
    F.col("v.transaction_date"),
    F.col("v.customer_id"),
    F.col("v.product_id") if "product_id" in df_enriquecido.columns else F.col("v.product_name"),
    F.col("v.quantity"),
    F.col("v.currency"),
    F.col("v.unit_price_local"),
    F.col("v.total_amount_local"),
    F.col("tasa_aplicada"),
    F.col("total_amount_usd")
)

# 🔍 VISUALIZACIÓN 3: Ver las tablas finales
print("--- TABLA DE HECHOS (FACT_VENTAS) ---")
display(fact_ventas.limit(5))

# Guardar en Capa Gold
fact_ventas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{BASE}/gold/fact_ventas")
dim_clientes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{BASE}/gold/dim_clientes")
dim_productos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{BASE}/gold/dim_productos")

print("✓ Capa Gold procesada correctamente.")

--- ESQUEMA: VENTAS SILVER ---
root
 |-- transaction_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- currency: string (nullable = true)
 |-- unit_price_local: double (nullable = true)
 |-- total_amount_local: double (nullable = true)

--- ESQUEMA: TASAS SILVER ---
root
 |-- moneda: string (nullable = true)
 |-- tipo_cambio_usd: double (nullable = true)
 |-- fecha_ingesta: timestamp (nullable = true)

--- VISTA PREVIA DEL CRUCE Y MONTO EN USD ---


transaction_id,currency,total_amount_local,tasa_aplicada,total_amount_usd
TXN-00001,JPY,255.8,1.0,255.8
TXN-00002,GBP,86.4,1.0,86.4
TXN-00003,CHF,1306.64,1.0,1306.64
TXN-00004,BRL,104.4,1.0,104.4
TXN-00005,BRL,323.57,1.0,323.57


--- TABLA DE HECHOS (FACT_VENTAS) ---


transaction_id,transaction_date,customer_id,product_name,quantity,currency,unit_price_local,total_amount_local,tasa_aplicada,total_amount_usd
TXN-00001,2024-01-29,CUST-0107,Ergonomic Chair,1,JPY,255.8,255.8,1.0,255.8
TXN-00002,2024-01-11,CUST-0121,USB-C Hub,2,GBP,43.2,86.4,1.0,86.4
TXN-00003,2024-01-22,CUST-0124,Monitor 27-inch,4,CHF,326.66,1306.64,1.0,1306.64
TXN-00004,2024-01-02,CUST-0121,Wireless Mouse,4,BRL,26.1,104.4,1.0,104.4
TXN-00005,2024-01-29,CUST-0128,Monitor 27-inch,1,BRL,323.57,323.57,1.0,323.57


✓ Capa Gold procesada correctamente.


In [0]:
# Agregación mensual/diaria para gráfico gerencial
df_kpi_gerencial = (
    fact_ventas
    .groupBy("transaction_date")
    .agg(F.sum("total_amount_usd").alias("total_ventas_usd"))
    .orderBy("transaction_date")
)
display(df_kpi_gerencial)

transaction_date,total_ventas_usd
2024-01-01,3896.86
2024-01-02,5234.219999999999
2024-01-03,5198.46
2024-01-04,1518.6200000000001
2024-01-05,3792.49
2024-01-06,131.09
2024-01-07,1996.3700000000001
2024-01-08,31.09
2024-01-09,138.26
2024-01-10,1594.7399999999998


Databricks visualization. Run in Databricks to view.

In [0]:
df_kpi_gerencial.write.mode("overwrite").saveAsTable("default.kpi_ventas_gerencial")